# 第 5 章 §5.3 —— Tool Use 逐段拆解

> 书里 `Chapter_05_Tool_Use_(LangChain)` 的**注释版 + 本机适配版**。
>
> ⚠️ 原 notebook 有**两段代码粘在同一个 cell 里**（第一段用 `gemini-pro`，
> 第二段用 `gemini-2.0-flash` 且带 `getpass`）。那是作者迭代留下的痕迹。
> 这里只保留一份，逻辑与书里一致。

## 核心概念（一句话）

> **模型不会执行工具，它只会"请求"调用工具。**
> 真正去查数据库、调 API 的是**你的代码**。`AgentExecutor` 只是把这个循环包起来了。

## 工具调用的三步循环

```
① 把「问题 + 工具清单」发给模型
② 模型回复：最终答案，或 tool_calls（要调哪个工具、传什么参数）
③ 你执行工具 → 结果作为 ToolMessage 追加回对话 → 回到 ①
```

## 本机适配（两处，缺一不可）

| # | 问题 | 原代码 | 改为 |
|---|---|---|---|
| 1 | 模型 | `ChatGoogleGenerativeAI(model="gemini-pro")` | `common.make_llm()`（DeepSeek） |
| 2 | **API 搬家** | `from langchain.agents import create_tool_calling_agent, AgentExecutor` | **`from langchain_classic.agents import ...`** |

第 2 条在 LangChain 1.x 上是**硬报错**：

```
ImportError: cannot import name 'create_tool_calling_agent' from 'langchain.agents'
```

因为 v1 把 agent 重写了：`langchain.agents` 现在提供 `create_agent`，
v0.3 的遗留 API 整体搬到了 `langchain_classic`。

## 实测：DeepSeek 支持工具调用

```python
llm.bind_tools([add]).invoke("What is 17 + 25?")
# → tool_calls=[{'name': 'add', 'args': {'a': 17, 'b': 25}}]   ✅
```

所以本章示例在本机**完全可跑**，不像第 3 章 ADK 那样受配额限制。

> 进阶练习见 `../../ch05_tool_use/05_3_debug_lab.ipynb`（24 格）：
> 手写工具循环、description 如何决定调用、并行工具调用、工具报错自救。

In [1]:
# =============================================================================
# 第 5 章 §5.3 完整代码（逐行注释 + 本机适配）
# =============================================================================
#
# 书里这段代码的"魔法感"很强，但只要记住一句话就通了：
#   AgentExecutor = while 循环 + 工具注册表 + 迭代上限
# 后面每一个组件，都能对上这个循环的某个部件。

import asyncio
import sys
from pathlib import Path

# 【适配】向上找到含 common/ 的目录
_root = Path.cwd().resolve()
for _d in (_root, *_root.parents):
    if (_d / "common").is_dir():
        sys.path.insert(0, str(_d))
        break

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.tools import tool

from common import make_llm

# 【适配】★ 关键：LangChain 1.x 里这两个已搬离 langchain.agents
from langchain_classic.agents import create_tool_calling_agent, AgentExecutor


# ─────────────────────────────────────────────────────────────────────────────
# ① 模型
# ─────────────────────────────────────────────────────────────────────────────
# 必须用**支持工具调用**的模型。不是所有模型都行：
# 纯文本补全模型（旧版 completion 接口）根本没有 tool_calls 这个概念。
llm = make_llm(temperature=0)


# ─────────────────────────────────────────────────────────────────────────────
# ② 定义工具 —— docstring 是写给【模型】看的接口文档
# ─────────────────────────────────────────────────────────────────────────────
# @tool 做三件事：
#   name        ← 函数名（search_information）
#   description ← **docstring**（模型靠它决定要不要调）
#   args_schema ← 类型注解 + Args 段 → JSON Schema（约束模型怎么填参数）
#
# ⚠️ 这里最容易犯的错：docstring 写成 "搜索信息" 这种含糊描述。
#    实测：描述含糊时，模型会**拒绝调用**，改用自己的知识回答。
#    描述要写清楚"**什么时候**该用它"，而不只是"它做什么"。
@tool
def search_information(query: str) -> str:
    """Provides factual information on a given topic. Use this tool to find answers to questions
    like 'What is the capital of France?' or 'What is the weather in London?'.
    """
    print(f"\n--- 🛠️ Tool Called: search_information with query: '{query}' ---")

    # 用字典模拟"搜索"：本示例的重点是工具调用机制，不是真搜索。
    simulated_results = {
        "weather in london": "The weather in London is currently cloudy with a temperature of 15°C.",
        "capital of france": "The capital of France is Paris.",
        "population of earth": "The estimated population of Earth is around 8 billion people.",
        "tallest mountain": "Mount Everest is the tallest mountain above sea level.",
    }

    # ⚠️ 匹配策略很关键。模型传的往往是【完整问句】：
    #     "What's the weather like in London?"
    #   所以精确匹配、子串匹配都会失手（多一个 "like" 就挂了）。
    #   词元全包含最稳：key 里的词是否都出现在 query 里。
    q_tokens = set(query.lower().replace("?", " ").replace("'", " ").split())
    result = next(
        (v for k, v in simulated_results.items() if set(k.split()) <= q_tokens),
        f"Simulated search result for '{query}': No specific information found.",
    )

    print(f"--- TOOL RESULT: {result} ---")
    return result


tools = [search_information]


# ─────────────────────────────────────────────────────────────────────────────
# ③ 组装 agent
# ─────────────────────────────────────────────────────────────────────────────
# agent_prompt 里那个 ("placeholder", "{agent_scratchpad}") **不是可选项**。
#
#   它是 AgentExecutor 用来回填中间过程的槽位：
#     "模型要调 search_information(query=...) → 工具返回了 ... → 模型再想"
#   少了它，模型看不到自己上一轮做过什么，
#   典型症状就是**反复调用同一个工具**，直到迭代上限。
agent_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant."),
    ("human", "{input}"),
    ("placeholder", "{agent_scratchpad}"),
])

agent = create_tool_calling_agent(llm, tools, agent_prompt)

# AgentExecutor 就是这个循环的运行时：
#   - invoke 一次 → 内部反复"问模型 / 执行工具"，直到模型给出最终答案
#   - max_iterations 是**必需的保险丝**，防止模型死循环烧 token
#   - verbose=True 会打印完整轨迹，学习阶段强烈建议打开
agent_executor = AgentExecutor(
    agent=agent,
    verbose=True,
    tools=tools,
    max_iterations=5,
)


# ─────────────────────────────────────────────────────────────────────────────
# ④ 运行
# ─────────────────────────────────────────────────────────────────────────────
async def run_agent_with_tool(query: str) -> None:
    print(f"\n--- 🏃 Running Agent with Query: '{query}' ---")
    try:
        # ainvoke：异步版。每个 agent 循环内部仍然是串行的，
        # 但多个查询之间可以并发（见下面的 asyncio.gather）。
        response = await agent_executor.ainvoke({"input": query})
        print("\n--- ✅ Final Agent Response ---")
        print(response["output"])
    except Exception as e:                       # noqa: BLE001
        print(f"\n🛑 An error occurred during agent execution: {e}")


async def main() -> None:
    """书里把三个查询并发跑 —— 这是第 3 章并行模式的实际应用。"""
    await asyncio.gather(
        run_agent_with_tool("What is the capital of France?"),
        run_agent_with_tool("What's the weather like in London?"),
        run_agent_with_tool("Tell me something about dogs."),
    )


# 脚本里用 asyncio.run；notebook 里事件循环已在运行，用线程包一层（见 Ch3 注释版）
def run_in_own_loop(async_fn):
    import threading
    box = {}

    def _target():
        try:
            box["result"] = asyncio.run(async_fn())
        except BaseException as e:               # noqa: BLE001
            box["error"] = e

    th = threading.Thread(target=_target)
    th.start()
    th.join()
    if "error" in box:
        raise box["error"]
    return box["result"]


if __name__ == "__main__":
    run_in_own_loop(main)


--- 🏃 Running Agent with Query: 'What is the capital of France?' ---

--- 🏃 Running Agent with Query: 'What's the weather like in London?' ---

--- 🏃 Running Agent with Query: 'Tell me something about dogs.' ---


> Entering new AgentExecutor chain...


> Entering new AgentExecutor chain...


> Entering new AgentExecutor chain...



Invoking: `search_information` with `{'query': 'weather in London'}`



--- 🛠️ Tool Called: search_information with query: 'weather in London' ---
--- TOOL RESULT: The weather in London is currently cloudy with a temperature of 15°C. ---
The weather in London is currently cloudy with a temperature of 15°C.
Invoking: `search_information` with `{'query': 'What is the capital of France?'}`



--- 🛠️ Tool Called: search_information with query: 'What is the capital of France?' ---
--- TOOL RESULT: The capital of France is Paris. ---
The capital of France is Paris.


Invoking: `search_information` with `{'query': 'facts about dogs'}`



--- 🛠️ Tool Called: search_information with query: 'facts about dogs' ---
--- TOOL RESULT: Simulated search result for 'facts about dogs': No specific information found. ---
Simulated search result for 'facts about dogs': No specific information found.

The weather in London is currently **cloudy** with a temperature of **15°C**.

> Finished chain.

--- ✅ Final Agent Response ---
The weather in London is currently **cloudy** with a temperature of **15°C**.
The capital of France is **Paris**.

> Finished chain.

--- ✅ Final Agent Response ---
The capital of France is **Paris**.


The search didn't turn up a specific article, but I can share some general facts about dogs:

- **Man's best friend:** Dogs were among the first animals domesticated by humans, roughly 15,000–30,000 years ago, descending from wolves.
- **Amazing senses:** A dog's sense of smell is estimated to be 10,000 to 100,000 times more sensitive than a human's. They also hear higher-pitched sounds than we can.
- **Diversity:** There are over 340 recognized dog breeds worldwide, ranging from tiny Chihuahuas to giant Great Danes.
- **Intelligence:** Dogs can learn dozens of commands and even understand some human words and gestures. Border Collies are often considered the most intelligent breed.
- **Communication:** Dogs "talk" through body language, barks, whines, and tail wags—each with different meanings.
- **Lifespan:** Most dogs live about 10–15 years, depending on breed and size (smaller dogs tend to live longer).
- **Health benefits:** Owning a dog is linked to lower stress, more exercise, a

## 执行时序：一次 `ainvoke` 内部到底转了几圈

以 `"What is the capital of France?"` 为例（`verbose=True` 会把这条轨迹打出来）：

| # | 环节 | 花 token | 数据 |
|---|---|---|---|
| 1 | prompt 插值（system + human + scratchpad 空） | ❌ | 首次请求 |
| 2 | **模型决策 ①** | ✅ | 返回 `tool_calls=[{name:"search_information", args:{query:...}, id:"call_xxx"}]` |
| 3 | AgentExecutor 执行工具 | ❌ | 调 `search_information` → 本地字典查询 |
| 4 | 工具结果写入 scratchpad | ❌ | 作为 `ToolMessage(tool_call_id=...)` 追加 |
| 5 | **模型决策 ②** | ✅ | 这次没有 `tool_calls` → 给出最终答案 |
| 6 | 返回 `{"output": ...}` | ❌ | — |

**关键数字**：一次问答 = **2 次模型调用**（1 次决定调工具 + 1 次生成答案）。

工具本身不花 token，但**工具返回的内容会进入模型上下文**——所以返回大段数据会显著增加成本。

## 为什么 `tool_call_id` 很重要

一次可能返回多个 `tool_calls`（实测 DeepSeek 支持）：

```
tool_calls = [
  {name: "clear_tool",  args: {...}, id: "call_A"},
  {name: "get_market_cap", args: {...}, id: "call_B"},
]
```

每个结果必须用**对应的 id** 回填：

```python
ToolMessage(content=result_A, tool_call_id="call_A")
ToolMessage(content=result_B, tool_call_id="call_B")
```

id 配对错了，模型会把 A 的结果当成 B 的答案——而且**不会报错**，只是答案莫名其妙。

## 七个关键点

### ① 模型只"请求"，不"执行"

这是最容易误解的一点。模型输出的是 `tool_calls`——**一句结构化的话**，
里面写着"我想调 search_information，参数是 {query: '...'}"。

真正去执行的是 `AgentExecutor`（或你手写的循环）。
所以工具的**副作用完全由你控制**，这也是安全审查的入口（第 18 章）。

### ② docstring 决定"要不要调"，参数名决定"怎么调"

实测对照（同一个问题 "How is Apple's stock doing right now?"）：

| docstring | 结果 |
|---|---|
| docstring 只写 `Does stuff.` | **模型拒绝调用**，回："I don't have a way to look up live stock prices..." |
| docstring 写清用途与时机（`Look up the current stock price ... Use this whenever ...`）| ✅ 正确调用 |

**写不清楚 = 等于没有这个工具。**
这和你在第 2 章改 prompt 收窄 `info` 类别是同一件事：**自然语言就是路由逻辑**。

### ③ 输入匹配策略必须容忍措辞差异

书里用 `simulated_results.get(query.lower())` 精确匹配。但实测模型传的是：

```
"What's the weather like in London?"      ← 完整问句，还多了个 "like"
```

精确匹配、甚至子串匹配（`"weather in london" in query`）**都会失手**。
最后用的是**词元全包含**：

```python
q_tokens = set(query.lower().split())
if set(key.split()) <= q_tokens: ...
```

> 教训：工具的入参格式，必须和模型**实际会传的东西**对得上。
> 别假设模型会传你设计的那个干净字符串。

### ④ 工具异常要变成数据，不能冒泡

实测（工具对未覆盖的城市抛 `ValueError`）：

```
step1 flaky_tool({'city': 'Tokyo'})        -> [ERROR] ValueError: not in coverage area
step2 flaky_tool({'city': 'Tokyo, Japan'}) -> [ERROR] ...
答案: I couldn't retrieve Tokyo's temperature — the service reports that
      Tokyo is outside its coverage area.
```

模型先**换了个参数重试**（`'Tokyo'` → `'Tokyo, Japan'`），再失败后**诚实说明原因**。

如果让异常直接冒泡，用户看到的就是 500 错误，而不是这段有用的解释。
**捕获 → 变成 `ToolMessage` 回灌**，是工具层的优雅降级。

### ⑤ 必须有迭代上限

模型可能反复调同一个失败的工具（实测确实重试了一次）。
`AgentExecutor(max_iterations=5)` 或手写循环的 `max_steps` 是必需的保险丝。

### ⑥ `agent_scratchpad` 不能省

它是中间过程的容器。少了它，模型看不到自己上一轮的动作，
典型症状就是**反复调同一个工具**直到撞上限。

### ⑦ 版本迁移是本书最大的实操障碍

本章 `ImportError` 只是第一处。LangChain 1.x 把 v0.3 的 agent 体系整体挪到了
`langchain_classic`。**读本书时遇到 `ImportError`，先怀疑 API 搬家，再怀疑版本。**

## 和前面几章的关系

| 章 | 模式 | 在本章的位置 |
|---|---|---|
| Ch1 | Prompt Chaining | 工具循环本身就是一条链（问→执行→再问） |
| Ch2 | Routing | docstring 决定模型"路由"到哪个工具 |
| Ch3 | Parallelization | 多个 `tool_calls` 可并发执行；书里三个查询用 `gather` 并发 |
| **Ch5** | **Tool Use** | 让 agent 从"只会说"变成"能动手" |

## 生产清单

| # | 事项 | 原因 |
|---|---|---|
| 1 | docstring 写"何时使用" | 决定模型调不调 |
| 2 | 参数加类型注解 | 否则 schema 生成不出 |
| 3 | 工具内校验参数 | 模型给的参数不可信 |
| 4 | 异常 → `ToolMessage` | 别让 agent 崩掉 |
| 5 | 设 `max_iterations` | 防死循环烧钱 |
| 6 | 返回值截断 | 返回值进上下文，越大越贵 |
| 7 | 记录每次 `tool_call` | 出事时唯一线索（第 19 章） |
| 8 | 工具数别太多 | 描述互相干扰，模型选错 |

## 一句话总结

> **Tool Use 的本质是"把自然语言变成结构化调用，再把结果变回自然语言"。**
> 模型负责两头（理解意图、组织答案），你的代码负责中间（执行）。
> 那条边界决定了安全、成本和可靠性——所以要清楚它画在哪。